## Import

In [6]:
# !pip install datasets transformers datasets scikit-learn #For Colab

## Import

In [3]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("No GPU found")

GPU available: True
GPU name: NVIDIA GeForce RTX 5060 Laptop GPU


In [4]:
# Basic libraries
import pandas as pd
import numpy as np
import torch
import inspect

# Hugging Face libraries
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# ML utilities
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

C:\Users\aaditya\Desktop\Deep Learning Project\project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load data

In [5]:
# Load CSV files
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
test_labels = pd.read_csv("test_labels.csv")

# Show data shape
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

# Show first rows
train_df.head()

# These are the six output labels for multi-label classification
target_cols = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

Train shape: (159571, 8)
Test shape: (153164, 2)


## Split training and validation data

In [6]:
# Split original training data into training and validation sets
train_data, val_data = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42
)

# Reset index
train_data = train_data.reset_index(drop=True)
val_data = val_data.reset_index(drop=True)

print("Training data:", train_data.shape)
print("Validation data:", val_data.shape)

Training data: (127656, 8)
Validation data: (31915, 8)


## Convert pandas DataFrame to Hugginf Face Dataset

In [7]:
# Convert pandas DataFrame to Hugging Face Dataset
train_dataset = Dataset.from_pandas(train_data)
val_dataset = Dataset.from_pandas(val_data)

print(train_dataset.column_names)

['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']


## Load DistilBERT tokenizer

In [8]:
# DistilBERT model name
model_name = "distilbert-base-uncased"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

## Tokenize text and prepare labels

In [9]:
def preprocess_data(batch):
    """
    This function:
    1. Tokenizes the comment text
    2. Creates one 'labels' column containing 6 labels
    """

    # Convert text into tokens
    encoding = tokenizer(
        batch["comment_text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    # Create labels for each comment
    labels = []

    # Loop through each comment in the batch
    for i in range(len(batch["comment_text"])):

        # Store the 6 labels for one comment
        one_comment_labels = []

        # Collect labels: toxic, severe_toxic, obscene, etc.
        for col in target_cols:
            one_comment_labels.append(float(batch[col][i]))

        # Add this comment's labels
        labels.append(one_comment_labels)

    # Add labels to the encoded data
    encoding["labels"] = labels

    return encoding

## Apply preprocessing

In [10]:
# Tokenize training dataset and remove old columns
train_dataset = train_dataset.map(
    preprocess_data,
    batched=True,
    remove_columns=train_dataset.column_names
)

# Tokenize validation dataset and remove old columns
val_dataset = val_dataset.map(
    preprocess_data,
    batched=True,
    remove_columns=val_dataset.column_names
)

Map: 100%|█████████████████████████████████████████████████████████████| 31915/31915 [00:01<00:00, 16269.58 examples/s]


In [11]:
# After preprocessing, we should only have these columns:
print(train_dataset.column_names)
print(val_dataset.column_names)

['input_ids', 'token_type_ids', 'attention_mask', 'labels']
['input_ids', 'token_type_ids', 'attention_mask', 'labels']


## Set dataset format for PyTorch

In [12]:
# Convert dataset columns to PyTorch tensors
train_dataset.set_format("torch")
val_dataset.set_format("torch")

## Load pretrained DistilBERT model

In [13]:
# Label ID to label name
id2label = {
    0: "toxic",
    1: "severe_toxic",
    2: "obscene",
    3: "threat",
    4: "insult",
    5: "identity_hate"
}

# Label name to label ID
label2id = {
    "toxic": 0,
    "severe_toxic": 1,
    "obscene": 2,
    "threat": 3,
    "insult": 4,
    "identity_hate": 5
}

# Load pretrained DistilBERT for multi-label classification
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=6,
    problem_type="multi_label_classification",
    id2label=id2label,
    label2id=label2id
)

C:\Users\aaditya\Desktop\Deep Learning Project\project\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aaditya\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|█████████████████████████████████████████████████████████████|

## Define evaluation metrics

In [14]:
def compute_metrics(eval_pred):
    """
    This function calculates:
    1. ROC-AUC
    2. Precision
    3. Recall
    4. F1-score
    """

    # Get model outputs and true labels
    logits, labels = eval_pred

    # Convert logits to probabilities using sigmoid
    probabilities = 1 / (1 + np.exp(-logits))

    # Convert probabilities to 0 or 1 using threshold 0.5
    predictions = (probabilities >= 0.5).astype(int)

    # ROC-AUC
    roc_auc = roc_auc_score(
        labels,
        probabilities,
        average="macro"
    )

    # Precision
    precision = precision_score(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    # Recall
    recall = recall_score(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    # F1-score
    f1 = f1_score(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    return {
        "roc_auc": roc_auc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

## Define training settings

In [21]:
# Basic training settings
args_dict = {
    "output_dir": "./distilbert-toxic-model",

    # Save model after each epoch
    "save_strategy": "epoch",

    # Train for 2 epochs
    "num_train_epochs": 2,

    # Batch size
    "per_device_train_batch_size": 16,
    "per_device_eval_batch_size": 16,

    # Learning rate
    "learning_rate": 2e-5,

    # Regularization
    "weight_decay": 0.01,

    # Load best model after training
    "load_best_model_at_end": True,

    # Use ROC-AUC to choose best model
    "metric_for_best_model": "roc_auc",
    "greater_is_better": True,

    # Disable wandb
    "report_to": "none"
}

# Some versions use eval_strategy, older versions use evaluation_strategy
training_args_params = inspect.signature(TrainingArguments.__init__).parameters

if "eval_strategy" in training_args_params:
    args_dict["eval_strategy"] = "epoch"
else:
    args_dict["evaluation_strategy"] = "epoch"


# Create TrainingArguments object
training_args = TrainingArguments(
    output_dir="./distilbert-toxic-model",

    eval_strategy="epoch",
    save_strategy="epoch",

    num_train_epochs=5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    learning_rate=2e-5,
    weight_decay=0.01,

    load_best_model_at_end=True,
    metric_for_best_model="roc_auc",
    greater_is_better=True,

    report_to="none",

    # Use mixed precision only if GPU is available
    fp16=torch.cuda.is_available()
)

## Create Trainer

In [22]:
# Create Hugging Face Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
# Start training
trainer.train()

Epoch,Training Loss,Validation Loss,Roc Auc,Precision,Recall,F1
1,0.027311,0.043573,0.989098,0.698148,0.591457,0.631649


Writing model shards: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  5.92it/s]


## Evaluate on validation data

In [ ]:
# Evaluate on validation set
val_results = trainer.evaluate()

# Print results
print(val_results)

In [ ]:
# Merge test comments with test labels
test_full = test_df.merge(test_labels, on="id")

# Remove rows where all labels are -1
# -1 means those rows were not used for scoring
test_full = test_full[test_full[target_cols].sum(axis=1) != -6]

# Reset index
test_full = test_full.reset_index(drop=True)

print("Usable test data:", test_full.shape)

In [ ]:
# Convert test DataFrame to Hugging Face Dataset
test_dataset = Dataset.from_pandas(test_full)

# Apply same preprocessing
test_dataset = test_dataset.map(
    preprocess_data,
    batched=True,
    remove_columns=test_dataset.column_names
)

# Convert to PyTorch format
test_dataset.set_format("torch")

print(test_dataset.column_names)